In [ ]:
import os

os.chdir("..")

In [ ]:
# from sklearn.preprocessing import SplineTransformer
# from sklearn.linear_model import Ridge
# from patsy import dmatrix
import bambi as bmb
from prprod_repl import ProdRepl
import polars as pl
import arviz_plots as azp
import arviz as az

azp.style.use("arviz-variat")


pr = ProdRepl()

In [ ]:
df = pr.reg_data()
df = df.with_columns(
    capital_synth_log=pl.col("capital_synth").log(),
    labour_log=pl.col("labour").log(),
    energy_log=pl.col("energy"),
).sort(["year", "month", "muni"])
df = df.to_pandas()
df.describe()

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az

# --- 1. Preprocessing ---
# Continuous time index for monthly data
df['time_idx'] = (df['year'] - df['year'].min()) * 12 + (df['month'] - 1)
time_vec = df['time_idx'].values[:, None]

df['muni_idx'], muni_labels = pd.factorize(df['muni'])
n_munis = len(muni_labels)
coords = {"muni": muni_labels, "obs": np.arange(len(df)), "feature": [0]}

# --- 2. The Final "Stress Test" Model ---
with pm.Model(coords=coords) as cobb_douglas_final:
    # Data containers
    muni_idx = pm.Data("muni_idx", df['muni_idx'], dims="obs")
    log_k = pm.Data("log_k", df['capital_synth_log'], dims="obs")
    log_l = pm.Data("log_l", df['labour_log'], dims="obs")
    log_y = pm.Data("log_y", df['energy_log'], dims="obs")
    t = pm.Data("t", time_vec, dims=("obs", "feature"))

    # --- A. Structural Elasticities (Hierarchical) ---
    # We keep these hierarchical as the physics of production (K and L) 
    # should be similar across the island.
    mu_beta_k = pm.Normal("mu_beta_k", mu=0.3, sigma=0.05)
    mu_beta_l = pm.Normal("mu_beta_l", mu=0.7, sigma=0.05)
    
    beta_k = pm.Normal("beta_k", mu_beta_k, 0.05, dims="muni")
    beta_l = pm.Normal("beta_l", mu_beta_l, 0.05, dims="muni")

    # --- B. Independent Growth Slopes (The Stress Test) ---
    # We use a wider sigma (0.1) and NO hierarchy. 
    # This allows each municipality to 'own' its own trend.
    muni_slope = pm.Normal("muni_slope", mu=0, sigma=0.1, dims="muni")

    # --- C. Intercepts (Hierarchical) ---
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)
    z_alpha = pm.Normal("z_alpha", 0, 1, dims="muni")
    alpha = pm.Deterministic("alpha", mu_alpha + z_alpha * sigma_alpha, dims="muni")

    # --- D. Disciplined HSGP (Island-Wide Shocks) ---
    # Captures 2021-2024 macro events (inflation, grid changes, policy)
    ls_gp = pm.InverseGamma("ls_gp", mu=12, sigma=3) 
    eta_gp = pm.HalfNormal("eta_gp", 0.1) 
    
    cov_func = eta_gp**2 * pm.gp.cov.ExpQuad(1, ls=ls_gp)
    gp = pm.gp.HSGP(m=[20], L=[time_vec.max() * 1.2], cov_func=cov_func)
    phi_t = gp.prior("phi_t", X=t, dims="obs")

    # --- E. Likelihood ---
    mu = (alpha[muni_idx] + 
          muni_slope[muni_idx] * t[:, 0] + 
          beta_k[muni_idx] * log_k + 
          beta_l[muni_idx] * log_l + 
          phi_t)
    
    sigma_eps = pm.Exponential("sigma_eps", 1.0)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_eps, observed=log_y, dims="obs")

    # --- F. Optimized Sampling ---
    idata = pm.sample(1000, tune=1500, target_accept=0.97, random_seed=42)

# --- 3. Final Results Extraction ---
print("--- Structural Parameters ---")
print(az.summary(idata, var_names=["mu_beta_k", "mu_beta_l", "eta_gp"]))

# Identifying the "Winners" and "Losers"
slopes = az.summary(idata, var_names=["muni_slope"])
top_growth = slopes.sort_values(by="mean", ascending=False).head(10)
bottom_growth = slopes.sort_values(by="mean", ascending=True).head(10)

print("\n--- Top 10 Municipalities by TFP Growth Trend ---")
print(top_growth[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

print("\n--- Bottom 10 Municipalities (Decline) ---")
print(bottom_growth[['mean', 'hdi_3%', 'hdi_97%', 'r_hat']])

In [ ]:
import pandas as pd
import arviz as az

# 1. Get the municipality names in the correct order used by the model
# Assuming 'municipio' was your original categorical column in 'df'
muni_names = df['muni'].astype('category').cat.categories

# 2. Generate the summary and reset the index to get the numeric IDs
slopes = az.summary(idata, var_names=["muni_slope"])
slopes = slopes.reset_index()

# 3. Map the IDs to the actual names
# This assumes muni_slope[0] corresponds to muni_names[0]
slopes['muni'] = muni_names

# 4. Create the final clean DataFrame
tfp_results = slopes[['muni', 'mean', 'hdi_3%', 'hdi_97%', 'r_hat']]
tfp_results.columns = ['muni', 'TFP_Trend', 'HDI_3', 'HDI_97', 'R_Hat']

# 5. Identify Winners and Losers
top_growth = tfp_results.sort_values(by="TFP_Trend", ascending=False).head(10)
bottom_growth = tfp_results.sort_values(by="TFP_Trend", ascending=True).head(10)

# Displaying results
print("--- Top 10 TFP Growth (Winners) ---")
print(top_growth.to_string(index=False))

print("\n--- Bottom 10 TFP Growth (Losers) ---")
print(bottom_growth.to_string(index=False))

In [ ]:
gdf = pr.county_geom()
import unicodedata

def clean_muni_string(s):
    if s is None: return None
    # Normalize to NFD and remove non-spacing marks (accents)
    s = unicodedata.normalize('NFD', str(s))
    s = "".join([c for c in s if not unicodedata.combining(c)])
    # Lowercase and replace spaces
    return s.lower().replace("  ", "_").replace(" ", "_")

# Apply to your GeoDataFrame
gdf['muni'] = gdf['name'].apply(clean_muni_string)
gdf

In [ ]:
merged_gdf = gdf.merge(
    tfp_results, 
    on='muni', 
    how='left'
)
merged_gdf.plot("TFP_Trend")

In [ ]:
import altair as alt

# 1. Prepare data
data_for_plot = merged_gdf.to_crs(epsg=4326)

# 2. Define the map
chart = alt.Chart(data_for_plot).mark_geoshape(
    stroke='white',
    strokeWidth=0.3 # Thinner lines look cleaner on large posters
).encode(
    color=alt.Color(
        'TFP_Trend:Q', 
        # 'viridis' is professional and color-blind friendly
        # Try 'magma' for a dark-to-bright purple/orange look
        # Try 'blueorange' if you want to emphasize the center (zero)
        scale=alt.Scale(scheme='viridis'), 
        legend=alt.Legend(
            title="Trend de TFP", 
            orient='right', 
            gradientLength=200,
            titleFontSize=12
        )
    ),
    tooltip=[
        alt.Tooltip('Municipio:N', title='Municipio'),
        alt.Tooltip('TFP_Trend:Q', title='Crecimiento TFP', format='.4f'),
        alt.Tooltip('HDI_3:Q', title='HDI (3%)', format='.4f'),
        alt.Tooltip('HDI_97:Q', title='HDI (97%)', format='.4f')
    ]
).properties(
    width=900, # Increased width for the 4-column poster layout
    height=500,
    title={
        "text": "Dinámica de la Productividad Total de los Factores",
        "subtitle": "Variación Espacial del Crecimiento de TFP (Estimados Bayesianos)",
        "fontSize": 22,
        "subtitleFontSize": 16,
        "anchor": "start",
        "color": "#006838",
        "offset": 20
    }
).configure_view(
    strokeWidth=0
).project(
    type='identity', reflectY=True 
)

chart.display()

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az

# --- 1. Preprocessing ---
def standardize(x):
    return (x - x.mean()) / x.std()

# Ensure your dataframe 'df' is loaded before this
df['k_std'] = standardize(df['capital_synth_log'])
df['l_std'] = standardize(df['labour_log'])
df['y_std'] = standardize(df['energy_log'])

# Time Indexing
df['time_idx'] = (df['year'] - df['year'].min()) * 12 + (df['month'] - 1)
t_raw = df['time_idx'].values[:, None]
t_scaled = (t_raw - t_raw.min()) / (t_raw.max() - t_raw.min())

# Municipality Indexing
df['muni_idx'], muni_labels = pd.factorize(df['muni'])
n_munis = len(muni_labels)

coords = {
    "muni": muni_labels, 
    "obs": np.arange(len(df)),
    "feature": [0]
}

# --- 2. Model Construction ---
with pm.Model(coords=coords) as cobb_douglas_gp_model:
    
    # Data Containers
    muni_idx = pm.Data("muni_idx", df['muni_idx'], dims="obs")
    log_k = pm.Data("log_k", df['k_std'], dims="obs")
    log_l = pm.Data("log_l", df['l_std'], dims="obs")
    log_y = pm.Data("log_y", df['y_std'], dims="obs")
    t = pm.Data("t", t_scaled, dims=("obs", "feature"))

    # --- A. Constant Returns to Scale (CRS) Elasticities ---
    # We estimate theta (capital share). Labour share is (1 - theta).
    # This prevents the 'elasticity collapse' you saw earlier.
    mu_theta_logit = pm.Normal("mu_theta_logit", mu=0, sigma=0.5) 
    sigma_theta = pm.HalfNormal("sigma_theta", 0.5)
    z_theta = pm.Normal("z_theta", 0, 1, dims="muni")
    
    # Deterministic transformation to [0, 1] space
    theta = pm.Deterministic("theta", pm.math.invlogit(mu_theta_logit + z_theta * sigma_theta), dims="muni")
    
    # --- B. Municipality Intercepts & Slopes ---
    alpha_muni = pm.Normal("alpha_muni", mu=0, sigma=1, dims="muni")
    
    sigma_slope = pm.HalfNormal("sigma_slope", 0.1)
    z_slope = pm.Normal("z_slope", 0, 1, dims="muni")
    muni_slope = pm.Deterministic("muni_slope", z_slope * sigma_slope, dims="muni")

    # --- C. Macro Trend (HSGP) ---
    # ls alpha=10, beta=5 forces a smoother, island-wide macro trend
    ls = pm.InverseGamma("ls", alpha=10, beta=5) 
    eta = pm.HalfNormal("eta", 0.1) 
    
    gp = pm.gp.HSGP(m=[15], L=[1.5], cov_func=eta**2 * pm.gp.cov.ExpQuad(1, ls=ls))
    phi_t = gp.prior("phi_t", X=t, dims="obs") 

    # --- D. The Cobb-Douglas Likelihood ---
    # log(Y) = alpha + slope*t + GP + [theta*log(K) + (1-theta)*log(L)]
    # Note: log_k and log_l are standardized, which is why alpha is centered at 0
    mu = (alpha_muni[muni_idx] + 
          muni_slope[muni_idx] * t[:, 0] + 
          phi_t +
          theta[muni_idx] * log_k + 
          (1 - theta[muni_idx]) * log_l)
    
    sigma_eps = pm.Exponential("sigma_eps", 1.0)
    # StudentT remains for outlier robustness in PR data
    nu = pm.Exponential("nu", 0.1) 
    y_obs = pm.StudentT("y_obs", nu=nu, mu=mu, sigma=sigma_eps, observed=log_y, dims="obs")

    # --- 3. Sampling ---
    # Increased max_treedepth to 12 to resolve the 'maximum tree depth' warnings.
    # Increased tuning to 2000 for better mass matrix adaptation.
    trace = pm.sample(
        draws=1000, 
        tune=2000, 
        target_accept=0.96, 
        max_treedepth=12, 
        random_seed=42
    )

# --- 4. Post-Estimation Analysis ---
# 1. Check Convergence
print(az.summary(trace, var_names=["mu_theta_logit", "eta", "ls"]))

# 2. Extract Elasticities
# Capital elasticity is theta, Labour is 1-theta
summary_df = az.summary(trace, var_names=["theta"])
print("\nTop 5 Municipalities by Capital Elasticity (theta):")
print(summary_df.sort_values(by="mean", ascending=False).head(5))

# 3. Extract Growth Rates (Slopes)
slopes_df = az.summary(trace, var_names=["muni_slope"])
print("\nTop 5 Municipalities by Growth Slope:")
print(slopes_df.sort_values(by="mean", ascending=False).head(5))

In [ ]:
# 3. Extract Growth Rates (Slopes)
slopes_df = az.summary(idata, var_names=["muni_slope"])
print("\nTop 5 Municipalities by Growth Slope:")
print(slopes_df.sort_values(by="mean", ascending=False).head(5))

In [ ]:
import matplotlib.pyplot as plt

# Plotting the 94% HDI for each municipality's slope
# We'll filter for a subset if there are too many, or plot all
az.plot_forest(idata, var_names=["muni_slope"], combined=True, figsize=(10, 12))
plt.axvline(0, color="red", linestyle="--", alpha=0.5)
plt.title("Municipal Growth Slopes (Hierarchical Estimates)")
plt.show()

In [ ]:
import arviz as az

# Update var_names to match the variables present in the new model
# We look at the means of the betas and the sigma of the intercepts
state_vars = ["mu_beta_k", "mu_beta_l", "sigma_alpha", "sigma_slope", "eta"]
state_results = az.summary(idata, var_names=state_vars)

print("--- State-Level Production Parameters ---")
print(state_results)

# To get specific elasticity values
means = state_results['mean'].to_dict()

# Note: These are in log-space because of the LogNormal/exp() transformation
# We can calculate the expected mean elasticity
import numpy as np
cap_elasticity = np.exp(means['mu_beta_k'])
lab_elasticity = np.exp(means['mu_beta_l'])

print(f"\nApprox. State Capital Elasticity: {cap_elasticity:.3f}")
print(f"Approx. State Labour Elasticity: {lab_elasticity:.3f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot the posterior distributions for the state-level elasticities
az.plot_posterior(idata, var_names=["mu_beta_k", "mu_beta_l"], 
                  ref_val=[0.3, 0.7], # Comparing against standard economic benchmarks
                  color='skyblue')
plt.suptitle("State-Level Elasticity Posteriors")
plt.show()

In [ ]:
import pymc as pm
import arviz as az

sigmas_to_test = [0.1, 0.5, 10.0]
results = {}

for s in sigmas_to_test:
    with pm.Model() as test_model:
        # Standardize your inputs for better comparison
        mu_beta_k = pm.Normal("mu_beta_k", mu=0.3, sigma=s)
        mu_beta_l = pm.Normal("mu_beta_l", mu=0.7, sigma=s)
        
        # ... (rest of your model code here) ...
        
        trace = pm.sample(1000, tune=1000, target_accept=0.9, progressbar=False)
        results[f"sigma_{s}"] = trace

# Compare the posteriors visually
az.plot_density(
    [results["sigma_0.1"], results["sigma_0.5"], results["sigma_10.0"]],
    var_names=["mu_beta_k", "mu_beta_l"],
    data_labels=["Strong (0.1)", "Moderate (0.5)", "Flat (10.0)"]
)

In [ ]:
import pandas as pd
import arviz as az

# List to store summary dataframes
summary_list = []

for label, trace in results.items():
    # Calculate summary statistics for the hyperpriors
    summary = az.summary(trace, var_names=["mu_beta_k", "mu_beta_l"], round_to=3)
    
    # Add a column to identify which prior sigma this belongs to
    summary["prior_sigma"] = label
    summary_list.append(summary)

# Combine all summaries into one table
final_comparison = pd.concat(summary_list)

# Reorder columns for readability
cols = ['prior_sigma', 'mean', 'sd', 'hdi_3%', 'hdi_97%', 'ess_bulk', 'r_hat']
final_comparison = final_comparison[cols]

print("--- Prior Sensitivity Numerical Comparison ---")
print(final_comparison)

# Optional: specifically print the shift in Returns to Scale
print("\n--- Returns to Scale (RTS) Drift ---")
for label, trace in results.items():
    k_mean = trace.posterior["mu_beta_k"].mean().values
    l_mean = trace.posterior["mu_beta_l"].mean().values
    print(f"{label}: RTS = {k_mean + l_mean:.3f}")

In [ ]:
import pandas as pd
import arviz as az

# 1. Extract the varying coefficients for all municipalities
muni_betas = az.summary(idata, var_names=["beta_k", "beta_l"], round_to=3)

# 2. Add the county names back to the summary
# muni_labels was the list of names from pd.factorize earlier
muni_betas['county'] = list(muni_labels) * 2 # One for beta_k, one for beta_l
muni_betas['parameter'] = (['Capital'] * n_munis) + (['Labour'] * n_munis)

# 3. Pivot for easy reading
pivot_betas = muni_betas.pivot(index='county', columns='parameter', values='mean')
pivot_betas['Total_RTS'] = pivot_betas['Capital'] + pivot_betas['Labour']

# Sort by the most capital-efficient counties
print(pivot_betas.sort_values(by='Capital', ascending=False).head(10))